Implementation of cuttle [cubic illuminance](https://www.researchgate.net/profile/Christopher-Cuttle/publication/274538293_Research_Note_A_practical_approach_to_cubic_illuminance_measurement/links/56a6d84b08ae997e22ba4fc6/Research-Note-A-practical-approach-to-cubic-illuminance-measurement.pdf)

In [2]:
#write the cubic points from the points.txt file --> done
import numpy as np
# make grid points for cubic illumination
class CubicfromPoints:
    "this"
    def __init__(self, input_file, space_angle=0, output_file=None):
        self.input_file = input_file
        self.space_angle = space_angle
        self.output_file = output_file or input_file.replace('points.txt', f'points_cubic_spaceAngle{space_angle}.txt')
        
        self.cubic_grid_dtype = np.dtype([ # only the data for the input grid
            ('point', np.float64, (3,)),       # (x, y, z)
            ('x_plus', np.float64, (3,)),      # x+
            ('x_minus', np.float64, (3,)),     # x-
            ('y_plus', np.float64, (3,)),      # y+
            ('y_minus', np.float64, (3,)),     # y-
            ('z_plus', np.float64, (3,)),      # z+
            ('z_minus', np.float64, (3,)),     # z-
            ('u_plus', np.float64, (3,)),      # u+
            ('u_minus', np.float64, (3,)),     # u-
            ('v_plus', np.float64, (3,)),      # v+
            ('v_minus', np.float64, (3,)),     # v-
            ('w_plus', np.float64, (3,)),      # w+
            ('w_minus', np.float64, (3,))      # w-
        ])
        
        self.rotation_matrix = self._compute_rotation_matrix()
        self.data_array = None
    def _compute_rotation_matrix(self):
        theta = -np.pi / 4
        phi = np.arctan(np.sqrt(2))

        R_y = np.array([
            [np.cos(theta), 0, np.sin(theta)],
            [0, 1, 0],
            [-np.sin(theta), 0, np.cos(theta)]
        ])
        
        R_x = np.array([
            [1, 0, 0],
            [0, np.cos(phi), -np.sin(phi)],
            [0, np.sin(phi), np.cos(phi)]
        ])
        
        return R_x @ R_y
    def _read_input(self):
        with open(self.input_file, 'r') as f:
            lines = f.readlines()
            data = []
            for line in lines:
                values = line.strip().split()
                x, y, z, *_ = map(float, values)
                
                x_plus = np.array([np.cos(self.space_angle), np.sin(self.space_angle), 0])
                x_minus = np.array([-np.cos(self.space_angle), -np.sin(self.space_angle), 0])
                y_plus = np.array([-np.sin(self.space_angle), np.cos(self.space_angle), 0])
                y_minus = np.array([np.sin(self.space_angle), -np.cos(self.space_angle), 0])
                z_plus = np.array([0, 0, 1])
                z_minus = np.array([0, 0, -1])
                
                u_plus = self.rotation_matrix @ x_plus
                u_minus = self.rotation_matrix @ x_minus
                v_plus = self.rotation_matrix @ y_plus
                v_minus = self.rotation_matrix @ y_minus
                w_plus = self.rotation_matrix @ z_plus
                w_minus = self.rotation_matrix @ z_minus
                
                data.append((np.array([x, y, z]), x_plus, x_minus, y_plus, y_minus, z_plus, z_minus,
                             u_plus, u_minus, v_plus, v_minus, w_plus, w_minus))      
        self.data_array = np.array(data, dtype=self.cubic_grid_dtype)
    def write_output(self):
        if self.data_array is None:
            self._read_input()
        with open(self.output_file, 'w') as f:
            for i in self.data_array:
                x = i['point'][0]
                y = i['point'][1]
                z = i['point'][2]
                f.write(f"{x} {y} {z} {i['u_plus'][0]} {i['u_plus'][1]} {i['u_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['u_minus'][0]} {i['u_minus'][1]} {i['u_minus'][2]}\n")
                f.write(f"{x} {y} {z} {i['v_plus'][0]} {i['v_plus'][1]} {i['v_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['v_minus'][0]} {i['v_minus'][1]} {i['v_minus'][2]}\n")
                f.write(f"{x} {y} {z} {i['w_plus'][0]} {i['w_plus'][1]} {i['w_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['w_minus'][0]} {i['w_minus'][1]} {i['w_minus'][2]}\n")

# -- postprocess -- calculate cubic illumination 
class cubic_illumination:
    

In [5]:
# example
E_u_plus = 10
E_u_minus = 230
E_v_plus = 320
E_v_minus = 50
E_w_plus = 40
E_w_minus = 530
CI = cubic_illumination(E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus)
print(CI.Ecl)

443.4042998137892


In [31]:
# calculate per-point cubic illumination from a tar.gz output file. 
import os
import numpy as np
import pandas as pd

result_path = os.path.join(r"../DelftBlueFiles/Outputs/s08_4_0_20240718_12_40_07_265209_1984586e-44f2-11ef-bad6-4c526248e5a3.csv.gz")
grid_path = os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S08/points_cubic_spaceAngle0.txt")

number_of_points_in_grid = 0
with open(grid_path, 'r') as f:
    for line in f:
        number_of_points_in_grid += 1
print("number of lines in grid file: ", number_of_points_in_grid)

number_of_points_in_result = 0
array = np.asarray(pd.read_csv(result_path, compression='gzip', header=None))
print("number of columns in result file: ", array.shape[1])

number of lines in grid file:  3714
number of columns in result file:  1416


In [4]:
#grid count match check
import os
import numpy as np
import pandas as pd
# Debug, plot three numbers: number of lines in grid file, number of columns in result file, and number of lines in regenrated grid line

non_cubic_grid_oath = os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S08/points.txt")
grid_path = os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S08/points_cubic_spaceAngle0.txt")
regenrated_grid_path = os.path.join(r"/scratch/nforouzandehsh/FormalizedLoD/regenerated_cubic.txt")
CubicfromPoints(os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S08/points.txt"), space_angle=0,output_file=regenrated_grid_path).write_output()
result_path = os.path.join(r"../DelftBlueFiles/Outputs/s08_4_0_20240718_12_40_07_265209_1984586e-44f2-11ef-bad6-4c526248e5a3.csv.gz")

number_of_points_in_grid_non_cubic = 0
with open(non_cubic_grid_oath, 'r') as f:
    for line in f:
        number_of_points_in_grid_non_cubic += 1
print("number of lines in non-cubic grid file*6: ", 6*number_of_points_in_grid_non_cubic)

number_of_points_in_grid = 0
with open(grid_path, 'r') as f:
    for line in f:
        number_of_points_in_grid += 1
print("number of lines in grid file: ", number_of_points_in_grid)

number_of_points_in_result = 0
array = np.asarray(pd.read_csv(result_path, compression='gzip', header=None))
print("number of columns in result file: ", array.shape[1])

number_of_points_in_regenrated_grid = 0
with open(regenrated_grid_path, 'r') as f:
    for line in f:
        number_of_points_in_regenrated_grid += 1
print("number of lines in regenrated grid file: ", number_of_points_in_regenrated_grid)

# remove regeneratd grid file
os.remove(regenrated_grid_path)

number of lines in non-cubic grid file*6:  1416
number of lines in grid file:  1416
number of columns in result file:  1416
number of lines in regenrated grid file:  1416


In [13]:
cubic_object = CubicIllumination_postprocess(grid_path, grid_type='cubic', result_path=result_path)
# (8760: hours of the year, 1416:)

cubic grid
(8760, 1416)


In [26]:
number_of_lines = 0
with open(grid_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    number_of_lines += 1
print("number of unique points is" , number_of_lines/6)
# 1416

number of unique points is 619.0


In [26]:
import numpy as np

class cuttle:
    """ 
    Implementation of "Research Note: A practical approach to cubic illuminance measurement" by Christopher Cuttle
    (https://www.researchgate.net/profile/Christopher-Cuttle/publication/274538293_Research_Note_A_practical_approach_to_cubic_illuminance_measurement/links/56a6d84b08ae997e22ba4fc6/Research-Note-A-practical-approach-to-cubic-illuminance-measurement.pdf)
    
    key features:
    0. generate cubic grid from points.txt file, 
    1. write the cubic grid to a file for radiance simulation
    2. postprocess the simulation results to calculate the cubic illuminance values for each point in the grid.
    
    Attributes:
        pointstxt_path (str): The file path to the grid file (points.txt).
        result_path (str): path to the simulated cubic csv.gz file.

    Methods:
        __init__(self, pointstxt_path, space_angle=0, cubic_grid_write_path=None):
            Initializes the cuttle object with the path to the points file, an optional space angle for grid generation, and an optional path for writing the cubic grid.
            
            Parameters:
                pointstxt_path (str): The file path to the points.txt file. This file should contain the 3D coordinates of points in space, which will be used to generate the cubic grid.
                space_angle (float, optional): The angle in degrees used to define the orientation of the cubic grid in space(to be further discussed). Defaults to 0, which means the grid will be aligned with the global coordinate system.
                cubic_grid_write_path (str, optional): The file path where the generated cubic grid should be written for Radiance simulation. If not provided, a default path will be used based on the input file name.
        
        _compute_rotation_matrix(self):
            Computes the rotation matrix based on the space angle provided during initialization.
            
        write_cubic_grid(self):
            Writes the generated cubic grid to a file for Radiance simulation.
        
        _per_point_calculation(simulated_node_array: np.ndarray):
            Calculates the cubic illumination values for each point in the grid based on the simulation results.
            
            Parameters:
                simulated_node_array (np.ndarray): An array with shape of (6, ) containing the simulated illuminance 
                values for a single point in the grid (E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus).
    """
    def __init__(self, pointstxt_path, space_angle=0):
        self.pointstxt_path = pointstxt_path
        self.space_angle = space_angle
        
        # self.cubic_grid_dtype = np.dtype([ # only the data for the input grid
        #     ('point', np.float64, (3,)),       # (x, y, z)
        #     ('x_plus', np.float64, (3,)),      # x+
        #     ('x_minus', np.float64, (3,)),     # x-
        #     ('y_plus', np.float64, (3,)),      # y+
        #     ('y_minus', np.float64, (3,)),     # y-
        #     ('z_plus', np.float64, (3,)),      # z+
        #     ('z_minus', np.float64, (3,)),     # z-
        #     ('u_plus', np.float64, (3,)),      # u+
        #     ('u_minus', np.float64, (3,)),     # u-
        #     ('v_plus', np.float64, (3,)),      # v+
        #     ('v_minus', np.float64, (3,)),     # v-
        #     ('w_plus', np.float64, (3,)),      # w+
        #     ('w_minus', np.float64, (3,))      # w-
        # ])
        # compute the rotation matrix 
        time_step_count = 8761
        self.cubic_grid_and_results_dtype = np.dtype([
            # Original point and illuminance values
            ('point', np.float64, (3,)),       # (x, y, z) coordinates 
            ('x_plus', np.float64, (3,)),      # x+ (unit vector)
            ('x_minus', np.float64, (3,)),     # x- (unit vector)
            ('y_plus', np.float64, (3,)),      # y+ (unit vector)
            ('y_minus', np.float64, (3,)),     # y- (unit vector)
            ('z_plus', np.float64, (3,)),      # z+ (unit vector)
            ('z_minus', np.float64, (3,)),     # z- (unit vector)
            ('u_plus', np.float64, (3,)),      # u+ (unit vector)
            ('u_minus', np.float64, (3,)),     # u- (unit vector)
            ('v_plus', np.float64, (3,)),      # v+ (unit vector)
            ('v_minus', np.float64, (3,)),     # v- (unit vector)
            ('w_plus', np.float64, (3,)),      # w+ (unit vector)
            ('w_minus', np.float64, (3,)),     # w- (unit vector)
            # Additional simulation outputs            
            ('E_u', np.float64, (time_step_count,)), # illumination vector magnitudes per time step
            ('E_v', np.float64, (time_step_count,)), # illumination vector magnitudes per time step
            ('E_w', np.float64, (time_step_count,)), # illumination vector magnitudes per time step
            ('~E_u', np.float64, (time_step_count,)), # symmetric components per time step
            ('~E_v', np.float64, (time_step_count,)), # symmetric components per time step
            ('~E_w', np.float64, (time_step_count,)), # symmetric components per time step
            ('e_u', np.float64, (time_step_count,)), # unit vectors per time step
            ('e_v', np.float64, (time_step_count,)), # unit vectors per time step
            ('e_w', np.float64, (time_step_count,)), # unit vectors per time step
            ('e_uvw', np.float64, (time_step_count,)), # NOT CALCULATED per time step
            ('e_x', np.float64, (time_step_count,)), # unit vectors per time step
            ('e_y', np.float64, (time_step_count,)), # unit vectors per time step
            ('e_z', np.float64, (time_step_count,)), # unit vectors per time step
            ('e_xyz', np.float64, (time_step_count,)), # NOT CALCULATED per time step
            ('|E|', np.float64, (time_step_count,)), # illumination vector magnitudes per time step
            ('Esr', np.float64, (time_step_count,)), # scalar illuminance per time step
            ('~E', np.float64, (time_step_count,)), # symmetric illuminance per time step
            ('|E|/Esr', np.float64, (time_step_count,)), # scalar illuminance per time step
            ('Ewp', np.float64, (time_step_count,)), # Horizontal working plane illuminance per time step
            ('Ecl', np.float64, (time_step_count,)), # mean cylindrical illuminance per time step
            ('Ecl/Ewp', np.float64, (time_step_count,)), # mean cylindrical illuminance per time step
            ])
        self.rotation_matrix = self._compute_rotation_matrix()
        
        # define empty array with dtype of self.cubic_grid_dtype
        self.cubic_grid_array = np.empty(0, dtype=self.cubic_grid_and_results_dtype)
        # Read points.txt file and generate the cubic grid_array
        with open(self.pointstxt_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                values = line.strip().split()
                x, y, z, *_ = map(float, values)
                
                x_plus = np.array([np.cos(self.space_angle), np.sin(self.space_angle), 0])
                x_minus = np.array([-np.cos(self.space_angle), -np.sin(self.space_angle), 0])
                y_plus = np.array([-np.sin(self.space_angle), np.cos(self.space_angle), 0])
                y_minus = np.array([np.sin(self.space_angle), -np.cos(self.space_angle), 0])
                z_plus = np.array([0, 0, 1])
                z_minus = np.array([0, 0, -1])
                
                u_plus = self.rotation_matrix @ x_plus
                u_minus = self.rotation_matrix @ x_minus
                v_plus = self.rotation_matrix @ y_plus
                v_minus = self.rotation_matrix @ y_minus
                w_plus = self.rotation_matrix @ z_plus
                w_minus = self.rotation_matrix @ z_minus
                
                initial_values = ( 
                    np.array([x, y, z]),  # point
                    x_plus, x_minus,      # x_plus, x_minus
                    y_plus, y_minus,      # y_plus, y_minus
                    z_plus, z_minus,      # z_plus, z_minus
                    u_plus, u_minus,      # u_plus, u_minus
                    v_plus, v_minus,      # v_plus, v_minus
                    w_plus, w_minus,      # w_plus, w_minus  The next values come from post process, what should be the initial values? it 
                    np.nan, np.nan, np.nan,        # E_u, E_v, E_w              --> np.nan as placeholder
                    np.nan, np.nan, np.nan,        # ~E_u, ~E_v, ~E_w           --> np.nan as placeholder
                    np.nan, np.nan, np.nan, np.nan,   # e_u, e_v, e_w, e_uvw    --> np.nan as placeholder
                    np.nan, np.nan, np.nan, np.nan,   # e_x, e_y, e_z, e_xyz    --> np.nan as placeholder
                    np.nan, np.nan, np.nan,        # |E|, Esr, ~E               --> np.nan as placeholder
                    np.nan, np.nan, np.nan,        # |E|/Esr, Ewp, Ecl          --> np.nan as placeholder
                    np.nan                   # Ecl/Ewp                          --> np.nan as placeholder
                )
                
                self.cubic_grid_array = np.append(
                    self.cubic_grid_array, np.array(initial_values, dtype=self.cubic_grid_and_results_dtype)
                )
                
        # self.grid_array = np.array(data, dtype=self.cubic_grid_dtype)
    def _compute_rotation_matrix(self):
        theta = -np.pi / 4
        phi = np.arctan(np.sqrt(2))

        R_y = np.array([
            [np.cos(theta), 0, np.sin(theta)],
            [0, 1, 0],
            [-np.sin(theta), 0, np.cos(theta)]
        ])
        
        R_x = np.array([
            [1, 0, 0],
            [0, np.cos(phi), -np.sin(phi)],
            [0, np.sin(phi), np.cos(phi)]
        ])
        
        return R_x @ R_y
    # public methods
    def write_cubic_grid(self, cubic_grid_write_path=None):
        self.cubic_grid_write_path = cubic_grid_write_path or self.pointstxt_path.replace('points.txt', f'points_cubic_spaceAngle{self.space_angle}.txt') # ugly

        with open(self.cubic_grid_write_path, 'w') as f:
            for i in self.cubic_grid_array:
                x = i['point'][0]
                y = i['point'][1]
                z = i['point'][2]
                f.write(f"{x} {y} {z} {i['u_plus'][0]} {i['u_plus'][1]} {i['u_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['u_minus'][0]} {i['u_minus'][1]} {i['u_minus'][2]}\n")
                f.write(f"{x} {y} {z} {i['v_plus'][0]} {i['v_plus'][1]} {i['v_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['v_minus'][0]} {i['v_minus'][1]} {i['v_minus'][2]}\n")
                f.write(f"{x} {y} {z} {i['w_plus'][0]} {i['w_plus'][1]} {i['w_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['w_minus'][0]} {i['w_minus'][1]} {i['w_minus'][2]}\n")
    # post process
    def post_process(self, raw_result_path):
        self.raw_result_path = raw_result_path
        # read the result file
        raw_result_array = np.asarray(pd.read_csv(result_path, compression='gzip', header=None))
        # make sure that the number of points in the grid and the result file match
        print(f"shape of the result file: {raw_result_array.shape}")
        print(f"number of unique points in the grid file: {self.cubic_grid_array.shape[0]}")
        
        result_point_count = raw_result_array.shape[1]
        cubic_grid_point_count = self.cubic_grid_array.shape[0] * 6

        if result_point_count != cubic_grid_point_count: 
            raise ValueError(f"The number of points in the grid file ({cubic_grid_point_count}) does not match the number of points in the result file ({result_point_count}).")
        print(f"shape of the result file: {raw_result_array.shape}")
        print(f"number of unique points in the grid file: {self.cubic_grid_array.shape[0]}")

        for ts in range(raw_result_array.shape[0]): # 8760
            for raw_result_sp in range(0, raw_result_array.shape[1], 6): # Process in steps of 6
                cubic_grid_array_sp = raw_result_sp // 6 # Index for cubic grid array
                # print(f"ts: {ts}, raw_result_sp:{raw_result_sp}, cubic_grid_array_sp: {cubic_grid_array_sp}")
                E_u_plus = raw_result_array[ts, raw_result_sp]
                E_u_minus = raw_result_array[ts, raw_result_sp+1]
                E_v_plus = raw_result_array[ts, raw_result_sp+2]
                E_v_minus = raw_result_array[ts, raw_result_sp+3]
                E_w_plus = raw_result_array[ts, raw_result_sp+4]
                E_w_minus = raw_result_array[ts, raw_result_sp+5]
                node_array = np.array([E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus])
                E_u, E_v, E_w, m_E_u, m_E_v, m_E_w, e_u, e_v, e_w, e_uvw, e_x, e_y, e_z, e_xyz, abs_E, Esr, m_E, abs_E_to_Esr, Ewp, Ecl, Ecl_Ewp = self._per_point_calculation(node_array) #
                print (f"for ts: {ts} and cubic_grid_array_sp : {cubic_grid_array_sp} illumination vector magnitude(abs_E):{abs_E} ")
                self.cubic_grid_array['E_u'][cubic_grid_array_sp] = np.append(self.cubic_grid_array['E_u'][cubic_grid_array_sp], E_u)
                self.cubic_grid_array['E_v'][cubic_grid_array_sp] = np.append(self.cubic_grid_array['E_v'][cubic_grid_array_sp], E_v)
                # self.cubic_grid_array['E_w'][cubic_grid_array_sp] = E_w
                # self.cubic_grid_array['~E_u'][cubic_grid_array_sp] = m_E_u
                # self.cubic_grid_array['~E_v'][cubic_grid_array_sp] = m_E_v
                # self.cubic_grid_array['~E_w'][cubic_grid_array_sp] = m_E_w
                # self.cubic_grid_array['e_u'][cubic_grid_array_sp] = e_u
                # self.cubic_grid_array['e_v'][cubic_grid_array_sp] = e_v
                # self.cubic_grid_array['e_w'][cubic_grid_array_sp] = e_w
                # self.cubic_grid_array['e_uvw'][cubic_grid_array_sp] = e_uvw
                # self.cubic_grid_array['e_x'][cubic_grid_array_sp] = e_x
                # self.cubic_grid_array['e_y'][cubic_grid_array_sp] = e_y
                # self.cubic_grid_array['e_z'][cubic_grid_array_sp] = e_z
                # self.cubic_grid_array['e_xyz'][cubic_grid_array_sp] = e_xyz
                # self.cubic_grid_array['|E|'][cubic_grid_array_sp] = abs_E
                # self.cubic_grid_array['Esr'][cubic_grid_array_sp] = Esr
                # self.cubic_grid_array['~E'][cubic_grid_array_sp] = m_E
                # self.cubic_grid_array['|E|/Esr'][cubic_grid_array_sp] = abs_E_to_Esr
                # self.cubic_grid_array['Ewp'][cubic_grid_array_sp] = Ewp
                # self.cubic_grid_array['Ecl'][cubic_grid_array_sp] = Ecl
                # self.cubic_grid_array['Ecl/Ewp'][cubic_grid_array_sp] = Ecl_Ewp 
                
            if len(self.cubic_grid_array) == 0:
                raise ValueError("The cubic grid array is empty. No calculations were performed.")
        return self.cubic_grid_array
    
    def _per_point_calculation(self, simulated_node_array: np.ndarray):
            if simulated_node_array.shape != (6,): # ensure the input array has the correct shape
                raise ValueError("The input array must have a shape of (6,). (E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus)")

            E_u_plus = simulated_node_array[0]
            E_u_minus = simulated_node_array[1]
            E_v_plus = simulated_node_array[2]
            E_v_minus = simulated_node_array[3]
            E_w_plus = simulated_node_array[4]
            E_w_minus = simulated_node_array[5]
            
            E_u = E_u_plus - E_u_minus
            E_v = E_v_plus - E_v_minus
            E_w = E_w_plus - E_w_minus
            # illumination vector magnitudes
            abs_E = np.sqrt(E_u**2 + E_v**2 + E_w**2) # |E|
            # symmetric components
            m_E_u = (E_u_plus + E_u_minus - E_u)/2 # ~E_u
            m_E_v = (E_v_plus + E_v_minus - E_v)/2 # ~E_v
            m_E_w = (E_w_plus + E_w_minus - E_w)/2 # ~E_w
            # symmetric illuminance
            m_E = (m_E_u + m_E_v + m_E_w)/3 # ~E
            # scalar illuminance 
            Esr = m_E + abs_E/4 # Esr
            # unit vectors
            if abs_E != 0 and not np.isnan(abs_E):
                e_u = E_u / abs_E
                e_v = E_v/abs_E
                e_w = E_w/abs_E
            else:
                e_u = np.nan # np.nan=not a number
                e_v = np.nan
                e_w = np.nan
            
            
            
            e_uvw = -999 # Should be calculated later
            e_x = 0.707*(e_v - e_w)
            e_y = 0.816*e_u - 0.408*(e_v + e_w)
            e_z = 0.577*(e_u + e_v + e_w)
            e_xyz = -999
            # Horizontal working plane illuminance
            if e_z < 0:
                Ewp = m_E
            else:
                Ewp = abs_E*e_z + m_E
            # mean cylindrical illuminance
            Ecl = (abs_E*(np.sqrt(e_x**2 + e_y**2))/np.pi) + m_E
            
            if Esr != 0 and not np.isnan(Esr):
                abs_E_to_Esr = abs_E/Esr
                Ecl_Ewp = Ecl/Ewp 
            else:
                abs_E_to_Esr = np.nan
                Ecl_Ewp = np.nan
                
            return E_u, E_v, E_w, m_E_u, m_E_v, m_E_w, e_u, e_v, e_w, e_uvw, e_x, e_y, e_z, e_xyz,abs_E, Esr, m_E, abs_E_to_Esr, Ewp, Ecl, Ecl_Ewp

In [25]:
# test
import os 
import numpy as np
import pandas as pd
pointstxt_path = os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S08/points.txt")
result_path = os.path.join(r"../DelftBlueFiles/Outputs/s08_4_0_20240718_12_40_07_265209_1984586e-44f2-11ef-bad6-4c526248e5a3.csv.gz")
cuttle_object = cuttle(pointstxt_path, space_angle=0)
cuttle_array = cuttle_object.post_process(result_path)

shape of the result file: (8761, 1416)
number of unique points in the grid file: 236
shape of the result file: (8761, 1416)
number of unique points in the grid file: 236
for ts: 0 and cubic_grid_array_sp : 0 illumination vector magnitude(abs_E):1.7320508075688772 


ValueError: could not broadcast input array from shape (8762,) into shape (8761,)

In [10]:
cuttle_array

array([([-21.613787,  13.17008 ,  15.929502], [1., 0., 0.], [-1., -0.,  0.], [-0.,  1.,  0.], [ 0., -1.,  0.], [0., 0., 1.], [ 0.,  0., -1.], [ 0.70710678, -0.57735027,  0.40824829], [-0.70710678,  0.57735027, -0.40824829], [0.        , 0.57735027, 0.81649658], [ 0.        , -0.57735027, -0.81649658], [-0.70710678, -0.57735027,  0.40824829], [ 0.70710678,  0.57735027, -0.40824829], 0., 0., 0., 0., 0., 0., nan, nan, nan, -999., nan, nan, nan, -999., 0., 0., 0., nan, nan, nan, nan),
       ([-21.364769,  12.404183,  15.929502], [1., 0., 0.], [-1., -0.,  0.], [-0.,  1.,  0.], [ 0., -1.,  0.], [0., 0., 1.], [ 0.,  0., -1.], [ 0.70710678, -0.57735027,  0.40824829], [-0.70710678,  0.57735027, -0.40824829], [0.        , 0.57735027, 0.81649658], [ 0.        , -0.57735027, -0.81649658], [-0.70710678, -0.57735027,  0.40824829], [ 0.70710678,  0.57735027, -0.40824829], 0., 0., 0., 0., 0., 0., nan, nan, nan, -999., nan, nan, nan, -999., 0., 0., 0., nan, nan, nan, nan),
       ([-21.115749,  11.638

In [22]:
# put this in an array
import os

result_path = os.path.join(r"../DelftBlueFiles/Outputs/s08_4_0_20240718_12_40_07_265209_1984586e-44f2-11ef-bad6-4c526248e5a3.csv.gz")
array = np.asarray(pd.read_csv(result_path, compression='gzip', header=None))
array[:, 0:6].shape

(8761, 6)